In [ ]:
import pandas as pd
from sklearn import neighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from scipy import stats

In [ ]:
cardio = pd.read_csv('cardio_v2.csv', sep=';')
cardio.shape
cardio = cardio.drop(columns=["id"])

print(cardio.head())

     age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  \
0  18393       2     168    62.0  110.0     80            1     1      0   
1  20228       1     156    85.0  140.0     -1            3     1      0   
2  18857       1     165    64.0  130.0     70            3     1      0   
3  17623       2     169    82.0  150.0    100            1     1      0   
4  17474       1     156    56.0  100.0     60            1     1      0   

   alco  active  cardio  
0     0       1       0  
1     0       1       1  
2     0       0       1  
3     0       1       1  
4     0       0       0  


In [ ]:
cardio['age'] = (cardio['age'] / 365.25).round(0).astype(int)

In [ ]:
print(cardio['ap_hi'].describe())

count    69999.000000
mean       128.816983
std        154.012499
min       -150.000000
25%        120.000000
50%        120.000000
75%        140.000000
max      16020.000000
Name: ap_hi, dtype: float64


In [ ]:
# Pressão sistólica normal: entre 60 e 250
cardio = cardio[(cardio['ap_hi'] >= 60) & (cardio['ap_hi'] <= 250)]

# Pressão diastólica normal: entre 40 e 150
cardio = cardio[(cardio['ap_lo'] >= 40) & (cardio['ap_lo'] <= 150)]

# Aplica Z-score
cols = ['height', 'weight', 'ap_hi', 'ap_lo']
for col in cols:
    media = cardio[col].mean()
    std = cardio[col].std()
    z_scores = np.abs((cardio[col] - media) / std)
    cardio[col] = cardio[col].where(z_scores <= 3, media)


In [ ]:
cardio['IMC'] = (cardio['weight'] / ((cardio['height'] / 100) ** 2)).round(2)

In [ ]:
print(cardio.head())

   age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  alco  \
0   50       2   168.0    62.0  110.0   80.0            1     1      0     0   
2   52       1   165.0    64.0  130.0   70.0            3     1      0     0   
3   48       2   169.0    82.0  150.0  100.0            1     1      0     0   
4   48       1   156.0    56.0  100.0   60.0            1     1      0     0   
5   60       1   151.0    67.0  120.0   80.0            2     2      0     0   

   active  cardio    IMC  
0       1       0  21.97  
2       0       1  23.51  
3       1       1  28.71  
4       0       0  23.01  
5       0       0  29.38  


In [ ]:
cardio.describe()


,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,IMC
count,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000,68753.000000
mean,53.291056,1.348756,164.422162,73.592539,125.858645,81.207009,1.364624,1.225867,0.088025,0.053569,0.803369,0.494786,27.261207
std,6.762619,0.476580,7.716870,13.157756,15.237029,9.044922,0.678901,0.571834,0.283334,0.225166,0.397454,0.499976,4.844944
min,30.000000,1.000000,140.000000,32.000000,80.000000,53.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,12.250000
25%,48.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,23.880000
50%,54.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,26.290000
75%,58.000000,2.000000,170.000000,81.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000,30.040000
max,65.000000,2.000000,189.000000,117.000000,176.000000,110.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,58.020000


In [ ]:
X = cardio.drop(columns=["cardio"])
X.head()



,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,IMC
0,50,2,168.0,62.0,110.0,80.0,1,1,0,0,1,21.97
2,52,1,165.0,64.0,130.0,70.0,3,1,0,0,0,23.51
3,48,2,169.0,82.0,150.0,100.0,1,1,0,0,1,28.71
4,48,1,156.0,56.0,100.0,60.0,1,1,0,0,0,23.01
5,60,1,151.0,67.0,120.0,80.0,2,2,0,0,0,29.38


In [ ]:
y = cardio["cardio"].values
print(y)

[0 1 1 ... 1 1 0]


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [ ]:
print(f"Treino:    {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validação: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Teste:     {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)")

Treino:    55002 (80%)
Validação: 6875 (10%)
Teste:     6876 (10%)


In [ ]:
melhor_k = 2
melhor_acc = 0

for k in range(2, 30):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_val, knn.predict(X_val))
    print(f"k={k:2d} | acurácia validação: {acc:.4f}")

    if acc > melhor_acc:
        melhor_acc = acc
        melhor_k = k

print(f"\nMelhor k: {melhor_k} | acurácia: {melhor_acc:.4f}")

k= 2 | acurácia validação: 0.6383
k= 3 | acurácia validação: 0.6679
k= 4 | acurácia validação: 0.6716
k= 5 | acurácia validação: 0.6812
k= 6 | acurácia validação: 0.6887
k= 7 | acurácia validação: 0.6945
k= 8 | acurácia validação: 0.7025
k= 9 | acurácia validação: 0.7066
k=10 | acurácia validação: 0.7052
k=11 | acurácia validação: 0.7097
k=12 | acurácia validação: 0.7113
k=13 | acurácia validação: 0.7130
k=14 | acurácia validação: 0.7094
k=15 | acurácia validação: 0.7165
k=16 | acurácia validação: 0.7155
k=17 | acurácia validação: 0.7169
k=18 | acurácia validação: 0.7185
k=19 | acurácia validação: 0.7181
k=20 | acurácia validação: 0.7174
k=21 | acurácia validação: 0.7162
k=22 | acurácia validação: 0.7164
k=23 | acurácia validação: 0.7183
k=24 | acurácia validação: 0.7191
k=25 | acurácia validação: 0.7206
k=26 | acurácia validação: 0.7196
k=27 | acurácia validação: 0.7206
k=28 | acurácia validação: 0.7200
k=29 | acurácia validação: 0.7213

Melhor k: 29 | acurácia: 0.7213


In [ ]:
clf = neighbors.KNeighborsClassifier(n_neighbors=5)

clf.fit(X_train, y_train)

KNeighborsClassifier()

In [ ]:
acc_teste = accuracy_score(y_test, clf.predict(X_test))
print(f"Acurácia no teste: {acc_teste:.4f}")

Acurácia no teste: 0.6870


In [ ]:
X_novos = [
    [50,    2,     168,    62,    110,    80,      1,          1,    0,     0,     1,    21.96],
    [55,    1,     156,    85,    140,    90,      3,          1,    0,     0,     1,    34.94],
    [52,    1,     165,    64,    130,    70,      3,          1,    0,     0,     0,    23.51],
    [48,    2,     169,    82,    150,    100,     1,          1,    0,     0,     1,    28.71],
    [48,    1,     156,    56,    100,    60,      1,          1,    0,     0,     0,    23.01],
    [60,    1,     151,    67,    120,    80,      2,          2,    0,     0,     0,    29.38],
]

predicao = clf.predict(X_novos)
for i in range(len(X_novos)):
    print(X_novos[i], "predição:", "Doença cardíaca" if predicao[i] == 1 else "Saudável")

[50, 2, 168, 62, 110, 80, 1, 1, 0, 0, 1, 21.96] predição: Saudável
[55, 1, 156, 85, 140, 90, 3, 1, 0, 0, 1, 34.94] predição: Doença cardíaca
[52, 1, 165, 64, 130, 70, 3, 1, 0, 0, 0, 23.51] predição: Saudável
[48, 2, 169, 82, 150, 100, 1, 1, 0, 0, 1, 28.71] predição: Doença cardíaca
[48, 1, 156, 56, 100, 60, 1, 1, 0, 0, 0, 23.01] predição: Saudável
[60, 1, 151, 67, 120, 80, 2, 2, 0, 0, 0, 29.38] predição: Saudável


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


In [ ]:
X = cardio.drop(columns=['cardio', 'height', 'weight'])

print(X.columns.tolist())

['age', 'gender', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'IMC']


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [ ]:
melhor_k = 2
melhor_acc = 0

for k in range(2, 30):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_val, knn.predict(X_val))
    print(f"k={k:2d} | acurácia validação: {acc:.4f}")

    if acc > melhor_acc:
        melhor_acc = acc
        melhor_k = k

print(f"\nMelhor k: {melhor_k} | acurácia: {melhor_acc:.4f}")

k= 2 | acurácia validação: 0.6396
k= 3 | acurácia validação: 0.6726
k= 4 | acurácia validação: 0.6817
k= 5 | acurácia validação: 0.6961
k= 6 | acurácia validação: 0.6993
k= 7 | acurácia validação: 0.7081
k= 8 | acurácia validação: 0.7088
k= 9 | acurácia validação: 0.7082
k=10 | acurácia validação: 0.7139
k=11 | acurácia validação: 0.7132
k=12 | acurácia validação: 0.7168
k=13 | acurácia validação: 0.7162
k=14 | acurácia validação: 0.7177
k=15 | acurácia validação: 0.7178
k=16 | acurácia validação: 0.7185
k=17 | acurácia validação: 0.7215
k=18 | acurácia validação: 0.7206
k=19 | acurácia validação: 0.7188
k=20 | acurácia validação: 0.7204
k=21 | acurácia validação: 0.7196
k=22 | acurácia validação: 0.7204
k=23 | acurácia validação: 0.7228
k=24 | acurácia validação: 0.7244
k=25 | acurácia validação: 0.7231
k=26 | acurácia validação: 0.7241
k=27 | acurácia validação: 0.7216
k=28 | acurácia validação: 0.7231
k=29 | acurácia validação: 0.7229

Melhor k: 24 | acurácia: 0.7244


In [ ]:
clf = KNeighborsClassifier(n_neighbors=melhor_k)
clf.fit(X_train, y_train)
acc_teste = accuracy_score(y_test, clf.predict(X_test))
print(f"Acurácia no teste: {acc_teste:.4f}")

Acurácia no teste: 0.7160
